#Library

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline, make_pipeline
import pandas as pd
import numpy as np
import joblib
import json
import seaborn as sns
from matplotlib import cm

In [ ]:
# Load Data
data = pd.read_csv('/content/child_data.csv')

#EDA

In [ ]:
data.describe()

,age,height,weight,bmi,bfp,gender,bmr
count,4028.000000,4028.000000,4028.000000,4028.000000,4028.000000,4028.000000,4028.000000
mean,8.803625,131.028438,35.517908,19.379506,13.640317,0.537488,1141.878323
std,5.344215,31.266922,18.100787,4.146274,6.231532,0.498655,320.101190
min,0.000000,65.120000,3.360443,6.633422,2.152477,0.000000,469.275238
25%,4.000000,107.800000,18.807625,16.214172,9.058462,0.000000,882.343115
50%,9.000000,137.554830,32.455000,18.806309,13.899925,1.000000,1134.298225
75%,13.000000,158.880000,52.712500,21.503717,16.862366,1.000000,1377.079248
max,18.000000,175.866529,73.348817,46.827848,27.348448,1.000000,1795.449713


In [ ]:
data.head()

,age,height,weight,bmi,bfp,gender,nutrition_status,bmr,food_preferences,health_conditions
0,0,69.96,10.29,21.024021,9.028825,1,Normal,561.95517,Non-Vegetarian,Sehat
1,0,71.69,10.89,21.189013,9.226815,1,Normal,578.29564,Non-Vegetarian,Anemia
2,0,68.54,10.93,23.266548,11.719858,1,Normal,563.71467,Vegetarian,Hypertension
3,0,69.69,9.38,19.313541,6.976249,1,Normal,548.46817,Non-Vegetarian,Hypertension
4,0,71.30,7.93,15.598906,2.518687,1,Gizi Buruk,536.76891,Vegetarian,Sehat


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4028 entries, 0 to 4027
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                4028 non-null   int64  
 1   height             4028 non-null   float64
 2   weight             4028 non-null   float64
 3   bmi                4028 non-null   float64
 4   bfp                4028 non-null   float64
 5   gender             4028 non-null   int64  
 6   nutrition_status   4028 non-null   object 
 7   bmr                4028 non-null   float64
 8   food_preferences   4028 non-null   object 
 9   health_conditions  4028 non-null   object 
dtypes: float64(5), int64(2), object(3)
memory usage: 314.8+ KB


In [ ]:
data['nutrition_status'].unique()

array(['Normal', 'Gizi Buruk', 'Berat badan lebih', 'Kurang Gizi',
       'Obesitas'], dtype=object)

In [ ]:
s = pd.Series(data['nutrition_status'])
s.value_counts()

,count
nutrition_status,
Normal,2257
Gizi Buruk,864
Kurang Gizi,602
Obesitas,227
Berat badan lebih,78


#Data Preprocessing

In [ ]:
# Encoding label
label_encoder = LabelEncoder()
data['Nutrition_Status_Encoded'] = label_encoder.fit_transform(data['nutrition_status'])

In [ ]:
# Fitur dan label
# features = ['age', 'gender', 'height', 'weight', 'bfp', 'bmr', 'bmi']
# X = data[features]
X = data.drop(columns=['Nutrition_Status_Encoded']).select_dtypes(include='number')
y = data['Nutrition_Status_Encoded'].values

In [ ]:
print(X.shape)
print(X.columns)

(4028, 7)
Index(['age', 'height', 'weight', 'bmi', 'bfp', 'gender', 'bmr'], dtype='object')


In [ ]:
# Pembagian data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Filter Method

In [ ]:
#Filter Method
mi = mutual_info_classif(X_train, y_train)
mi_series = pd.Series(mi, index=X.columns)
selected_filter = mi_series[mi_series > 0.05].index.tolist()
print("Fitur terpilih Filter Method:", selected_filter)

Fitur terpilih Filter Method: ['age', 'height', 'weight', 'bmi', 'bfp', 'gender', 'bmr']


Embedded Method

In [ ]:
#Embedded Method
pipe_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LogisticRegression(penalty='l1', solver='saga', C=0.1, max_iter=5000, random_state=42))
])
pipe_lasso.fit(X_train, y_train)

coef = pd.Series(np.abs(pipe_lasso.named_steps['lasso'].coef_).mean(axis=0), index=X.columns)
selected_embedded = coef[coef > 0].index.tolist()
print("Fitur terpilih Embedded Method:", selected_embedded)

Fitur terpilih Embedded Method: ['height', 'weight', 'bmi', 'gender', 'bmr']


Wrapper Method

In [ ]:
#Random Forest RFE
rf = RandomForestClassifier(random_state=42)
rfe_rf = RFE(estimator=rf, n_features_to_select=5)
rfe_rf.fit(X_train, y_train)
selected_wrapper_rf = X.columns[rfe_rf.support_].tolist()
print("Fitur terpilih Wrapper Method:", selected_wrapper_rf)

Fitur terpilih Wrapper Method: ['age', 'height', 'weight', 'bmi', 'bmr']


In [ ]:
#Linear SVM RFE
svc = LinearSVC(penalty='l2', dual=False, random_state=42, max_iter=5000)
rfe = RFE(estimator=svc, n_features_to_select=5, step=1)
pipeline = Pipeline([ ('scaler', StandardScaler()), ('rfe', rfe)])
pipeline.fit(X_train, y_train)

selected_wrapper_svc = X.columns[pipeline.named_steps['rfe'].support_].tolist()
print("Fitur terpilih Wrapper Method (SVC):", selected_wrapper_svc)

Fitur terpilih Wrapper Method (SVC): ['height', 'weight', 'bmi', 'bfp', 'gender']


In [ ]:
summary = pd.DataFrame({
    'Feature': X.columns,
    'Filter': ['✅' if f in selected_filter else '❌' for f in X.columns],
    'Embedded': ['✅' if f in selected_embedded else '❌' for f in X.columns],
    'Wrapper_RF': ['✅' if f in selected_wrapper_rf else '❌' for f in X.columns],
    'Wrapper_SVC': ['✅' if f in selected_wrapper_svc else '❌' for f in X.columns]
}).set_index('Feature')

print("\nSummary Feature Selection:")
print(summary)


Summary Feature Selection:
        Filter Embedded Wrapper_RF Wrapper_SVC
Feature                                       
age          ✅        ❌          ✅           ❌
height       ✅        ✅          ✅           ✅
weight       ✅        ✅          ✅           ✅
bmi          ✅        ✅          ✅           ✅
bfp          ✅        ❌          ❌           ✅
gender       ✅        ✅          ❌           ✅
bmr          ✅        ✅          ✅           ❌


In [ ]:
feature_votes = (
    summary.eq('✅')
    .astype(int)
    .sum(axis=1)
)

selected_features_final = feature_votes[feature_votes >= 3].index.tolist()
print("Fitur akhir terpilih:", selected_features_final)

Fitur akhir terpilih: ['height', 'weight', 'bmi', 'gender', 'bmr']


#Train Model

**skema k-fold cross-validation**

In [ ]:
# Skema 5-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Gunakan fitur hasil voting
X_train_fs = X_train[selected_features_final]
X_val_fs = X_val[selected_features_final]

**Support Vector Machine (SVM)**

In [ ]:
# Final Scaling (PENTING Untuk SVM!!!)
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(random_state=42))
])

param_grid_svm = {
    'svm__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf', 'poly'],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'svm__class_weight': [None, 'balanced'],
    'svm__degree': [2, 3, 4]
}

grid_svm = GridSearchCV(
    pipe_svm,
    param_grid_svm,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1
)

grid_svm.fit(X_train_fs, y_train)

print("Best SVM Params:", grid_svm.best_params_)
print("Best CV Accuracy:", grid_svm.best_score_)

Best SVM Params: {'svm__C': 100, 'svm__class_weight': None, 'svm__degree': 2, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Best CV Accuracy: 0.9913440221374877


**Random Forest**

In [ ]:
# param_grid_rf = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [None, 5, 10, 15, 20, 30],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': ['sqrt', 'log2', None],
#     'class_weight': ['balanced', 'balanced_subsample', None],
#     'bootstrap': [True, False]
# }

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None],
    'max_features': ['sqrt'],
}

rf = RandomForestClassifier(random_state=42)

grid_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train_fs, y_train)  # RF tidak wajib scaling

print("Best RF Params:", grid_rf.best_params_)
print("Best CV Accuracy:", grid_rf.best_score_)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best RF Params: {'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best CV Accuracy: 0.9841250755313442


**XGBoost**

In [ ]:
# param_grid_xgb = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [3, 5, 7, 9],
#     'learning_rate': [0.01, 0.05, 0.1],
#     'subsample': [0.7, 0.8, 0.9, 1.0],
#     'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
#     'gamma': [0, 0.1, 0.2],
#     'reg_alpha': [0, 0.01, 0.1],
#     'reg_lambda': [1, 1.5, 2]
# }

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42
)

grid_xgb = GridSearchCV(
    xgb,
    param_grid_xgb,
    cv=skf,
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1
)

sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

grid_xgb.fit(X_train_fs, y_train, sample_weight=sample_weights) # XGB tidak wajib scaling

print("Best XGB Params:", grid_xgb.best_params_)
print("Best XGB CV Accuracy:", grid_xgb.best_score_)

Fitting 5 folds for each of 15552 candidates, totalling 77760 fits


In [ ]:
# Model
models = {
    'SVM': grid_svm.best_estimator_,
    'Random Forest': grid_rf.best_estimator_,
    'XGBoost': grid_xgb.best_estimator_
}

In [ ]:
#Evaluasi semua model
for name, model in models.items():
    y_pred = model.predict(X_val_fs)
    acc = balanced_accuracy_score(y_val, y_pred)
    print(f"\n{name} Validation Accuracy: {acc:.4f}")
    print(classification_report(y_val, y_pred))


SVM Validation Accuracy: 0.9850
              precision    recall  f1-score   support

           0       0.89      1.00      0.94        16
           1       0.99      0.99      0.99       173
           2       0.99      0.97      0.98       120
           3       1.00      1.00      1.00       452
           4       1.00      0.96      0.98        45

    accuracy                           0.99       806
   macro avg       0.97      0.98      0.98       806
weighted avg       0.99      0.99      0.99       806


Random Forest Validation Accuracy: 0.9831
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        16
           1       1.00      1.00      1.00       173
           2       1.00      1.00      1.00       120
           3       1.00      1.00      1.00       452
           4       1.00      0.98      0.99        45

    accuracy                           1.00       806
   macro avg       0.99      0.98      0.99       806
w

In [ ]:
#Bandingkan Performa
results = {}
for name, model in models.items():
    y_pred = model.predict(X_val_fs)
    results[name] = {
        'accuracy': accuracy_score(y_val, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_val, y_pred),
        'confusion_matrix': confusion_matrix(y_val, y_pred)
    }

# Tampilkan perbandingan
comparison_df = pd.DataFrame(results).T
print(comparison_df[['accuracy', 'balanced_accuracy']])

               accuracy balanced_accuracy
SVM            0.992556          0.984955
Random Forest  0.997519          0.983056
XGBoost        0.990074          0.979558


#Save Model

In [ ]:
#Pilih Model Terbaik
best_model_name = max(results, key=lambda x: results[x]['balanced_accuracy'])
best_model = models[best_model_name]

print(f"\nModel terbaik: {best_model_name}")
print(f"Balanced Accuracy: {results[best_model_name]['balanced_accuracy']:.4f}")

# Simpan model terbaik
# joblib.dump(best_model, f'best_model_{best_model_name}.pkl')
# joblib.dump(label_encoder, 'label_encoder.pkl')


Model terbaik: SVM
Balanced Accuracy: 0.9850


In [ ]:
# Cek apakah model sudah termasuk preprocessing
print("Tipe model:", type(best_model))
print("Apakah Pipeline?", hasattr(best_model, 'steps'))

if hasattr(best_model, 'steps'):
    print("Steps dalam Pipeline:")
    for step_name, step in best_model.steps:
        print(f"- {step_name}: {type(step).__name__}")

Tipe model: <class 'sklearn.pipeline.Pipeline'>
Apakah Pipeline? True
Steps dalam Pipeline:
- scaler: StandardScaler
- svm: SVC


In [ ]:
metadata = {
    "model_name": best_model_name,
    "model_type": "sklearn.pipeline.Pipeline",
    "pipeline_steps": [
        {"name": name, "type": type(step).__name__}
        for name, step in best_model.steps
    ],
    "balanced_accuracy": float(results[best_model_name]['balanced_accuracy']),
    "input_features": best_model['svm'].n_features_in_ if hasattr(best_model['svm'], 'n_features_in_') else "unknown"
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)
print("✅ Metadata disimpan: model_metadata.json")

✅ Metadata disimpan: model_metadata.json
